<a href="https://colab.research.google.com/github/cpython-projects/python_da_17_03_26/blob/main/lesson_20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔷 ABC-аналіз

## Навіщо робити ABC-аналіз?

ABC-аналіз — це **простий і потужний спосіб розставити пріоритети**. Він допомагає відповісти на питання:

> Які товари приносять найбільшу частку виручки і де варто зосередити зусилля?

Ми майже завжди стикаємося з тим, що:

* **20% товарів** приносять **80% виручки** (принцип Парето),
* А більшість товарів створюють **мізерну частку доходу**, але потребують ресурсів (зберігання, логістика, маркетинг).

---

## Застосування

* управління запасами (логістика),
* оптимізація асортименту (retail),
* аналіз виручки (фінанси),
* управління SKU (Stock Keeping Unit/товарна позиція) (продуктова аналітика).

---

## Основна ідея

Поділити товари за **внеском у сукупну виручку**:

| Клас | Виручка | Орієнтовний % товарів | Ціль управління           |
| ---- | ------- | --------------------- | ------------------------- |
| A    | 70–80%  | 10–20%                | Максимальний контроль     |
| B    | 15–25%  | \~30%                 | Пошук покращень           |
| C    | ≤ 5%    | 50–60%                | Автоматизація, скорочення |

---

## Етапи ABC-аналізу

1. **Підготовка даних**: виручка по кожному товару (або SKU).
2. **Агрегація** (якщо потрібно): групування за ID, сума виручки.
3. **Сортування** за спаданням виручки.
4. **Накопичення** частки від загального обсягу.
5. **Класифікація** за порогами: A / B / C.
6. **Інтерпретація та дії**.


## Приклад

In [2]:
import pandas as pd

data = {
    'item_id': ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A7'],
    'total_sales_value': [1000, 800, 500, 200, 150, 100, 50, 2000]
}

df = pd.DataFrame(data)
df

,item_id,total_sales_value
0,A1,1000
1,A2,800
2,A3,500
3,A4,200
4,A5,150
5,A6,100
6,A7,50
7,A7,2000


In [3]:
# Крок 1. Групування за товарами (якщо дублюються)
df_grouped = df.groupby('item_id').agg({
    'total_sales_value': 'sum'
}).reset_index()
df_grouped

,item_id,total_sales_value
0,A1,1000
1,A2,800
2,A3,500
3,A4,200
4,A5,150
5,A6,100
6,A7,2050


In [4]:
# Крок 2. Сортування за спаданням
df_sorted = df_grouped.sort_values('total_sales_value', ascending=False).reset_index()
df_sorted

,index,item_id,total_sales_value
0,6,A7,2050
1,0,A1,1000
2,1,A2,800
3,2,A3,500
4,3,A4,200
5,4,A5,150
6,5,A6,100


In [5]:
df_sorted['total_sales_value'].cumsum()

,total_sales_value
0,2050
1,3050
2,3850
3,4350
4,4550
5,4700
6,4800


In [6]:
# Крок 3. Накопичення та відсоток

df_sorted['cum_total'] = df_sorted['total_sales_value'].cumsum()
df_sorted

,index,item_id,total_sales_value,cum_total
0,6,A7,2050,2050
1,0,A1,1000,3050
2,1,A2,800,3850
3,2,A3,500,4350
4,3,A4,200,4550
5,4,A5,150,4700
6,5,A6,100,4800


In [7]:
total = df_sorted.total_sales_value.sum()


In [8]:
total

np.int64(4800)

In [9]:
df_sorted['cum_total_percents'] = df_sorted.cum_total / total
df_sorted

,index,item_id,total_sales_value,cum_total,cum_total_percents
0,6,A7,2050,2050,0.427083
1,0,A1,1000,3050,0.635417
2,1,A2,800,3850,0.802083
3,2,A3,500,4350,0.906250
4,3,A4,200,4550,0.947917
5,4,A5,150,4700,0.979167
6,5,A6,100,4800,1.000000


In [12]:
# Крок 4. Присвоєння класу
def abc_classify(item):
  if item <= 0.8:
    return 'A'
  if item <= 0.95:
    return 'B'
  return 'C'

df_sorted['ABC'] = df_sorted.cum_total_percents.apply(abc_classify)
df_sorted

,index,item_id,total_sales_value,cum_total,cum_total_percents,ABC
0,6,A7,2050,2050,0.427083,A
1,0,A1,1000,3050,0.635417,A
2,1,A2,800,3850,0.802083,B
3,2,A3,500,4350,0.906250,B
4,3,A4,200,4550,0.947917,B
5,4,A5,150,4700,0.979167,C
6,5,A6,100,4800,1.000000,C


## Що робити після ABC-класифікації?

### Клас A (ключові)

* Максимум уваги: постачання, склад, просування.
* Стежити за залишками, рекламою, знижками.
* Часто аналізувати динаміку.

### Клас B (середня значущість)

* Оптимізувати: умови закупівлі, постачальників.
* Є потенціал зростання — варто подумати про маркетинг.

### Клас C (низькопріоритетні)

* Мінімізувати запаси.
* Об’єднувати SKU, видаляти неефективні.
* Можливо — перевести на автоматичний контроль.


# 🔷 XYZ-аналіз

## Навіщо робити XYZ-аналіз?

Якщо ABC-аналіз показує, **що важливо за виручкою**, то XYZ-аналіз показує:

> Наскільки **стабільний попит** на товар?

Ви можете мати товар із високою виручкою, але якщо він продається стрибкоподібно і нестабільно — з ним треба поводитися обережно (ризик залишків, дефіциту, списань).

---

## Основна ідея

Оцінити **прогнозованість попиту** за коефіцієнтом варіації.
*Коефіцієнт варіації* — це відносна міра розкиду даних, яка показує, наскільки сильно варіюються значення відносно їхнього середнього.

| Клас | Стабільність попиту     | Інтерпретація                      |
| ---- | ----------------------- | ---------------------------------- |
| X    | Висока стабільність     | Попит майже не змінюється          |
| Y    | Середня прогнозованість | Є коливання, але в межах норми     |
| Z    | Низька стабільність     | Хаотичний, сезонний або випадковий |

---

## Як вимірюється?

Через **коефіцієнт варіації (CV)**:

$$
\text{CV} = \frac{\text{Стандартне відхилення}}{\text{Середнє значення}}
$$

---

## Етапи XYZ-аналізу

1. Підготовка даних: продажі за періодами (наприклад, за місяцями).
2. Розрахунок середнього та стандартного відхилення по кожному товару.
3. Розрахунок коефіцієнта варіації (CV).
4. Класифікація на X/Y/Z.
5. Інтерпретація та дії.


## Приклад

In [13]:
data = {
    'item_id': ['A1', 'A2', 'A3', 'A4', 'A5'],
    'jan': [100, 120, 90, 200, 10],
    'feb': [100, 80, 100, 250, 30],
    'mar': [100, 110, 105, 50, 90],
    'apr': [100, 95, 95, 300, 70]
}

df = pd.DataFrame(data)
df

,item_id,jan,feb,mar,apr
0,A1,100,100,100,100
1,A2,120,80,110,95
2,A3,90,100,105,95
3,A4,200,250,50,300
4,A5,10,30,90,70


In [14]:
# Крок 1. Розрахунок середнього та стандартного відхилення
df['mean_sales'] = df[['jan', 'feb', 'mar', 'apr']].mean(axis=1)
df['std_sales'] = df[['jan', 'feb', 'mar', 'apr']].std(axis=1)

# Розраховуємо середнє та стандартне відхилення за місяцями


df

,item_id,jan,feb,mar,apr,mean_sales,std_sales
0,A1,100,100,100,100,100.00,0.000000
1,A2,120,80,110,95,101.25,17.500000
2,A3,90,100,105,95,97.50,6.454972
3,A4,200,250,50,300,200.00,108.012345
4,A5,10,30,90,70,50.00,36.514837


In [16]:
# Крок 2. Коефіцієнт варіації
df['cv'] = df.std_sales / df.mean_sales
df

,item_id,jan,feb,mar,apr,mean_sales,std_sales,cv
0,A1,100,100,100,100,100.00,0.000000,0.000000
1,A2,120,80,110,95,101.25,17.500000,0.172840
2,A3,90,100,105,95,97.50,6.454972,0.066205
3,A4,200,250,50,300,200.00,108.012345,0.540062
4,A5,10,30,90,70,50.00,36.514837,0.730297


In [18]:
# Крок 3. Класифікація за порогами
def xyz_classify(item):
  if item <= 0.25:
    return 'X'
  if item <= 0.50:
    return 'Y'
  return 'Z'


df['xyz'] = df.cv.apply(xyz_classify)
df

,item_id,jan,feb,mar,apr,mean_sales,std_sales,cv,xyz
0,A1,100,100,100,100,100.00,0.000000,0.000000,X
1,A2,120,80,110,95,101.25,17.500000,0.172840,X
2,A3,90,100,105,95,97.50,6.454972,0.066205,X
3,A4,200,250,50,300,200.00,108.012345,0.540062,Z
4,A5,10,30,90,70,50.00,36.514837,0.730297,Z


## Як інтерпретувати?

| Клас | Що це означає                              | Що робити?                                          |
| ---- | ------------------------------------------ | --------------------------------------------------- |
| X    | Попит стабільний, легко прогнозувати       | Планувати точно, тримати у постійній наявності      |
| Y    | Середня прогнозованість, помірні коливання | Підлаштовуватися під тренди, враховувати сезонність |
| Z    | Нестабільний, хаотичний попит              | Мінімальні запаси, уникати автоматичного поповнення |


# 🔷 ABC+XYZ матриця

## Комбінація ABC і XYZ: матриця 3×3

Об’єднуючи обидва аналізи, отримуємо **матрицю з 9 категорій товарів**:

|       | **X (стабільний)** | **Y (помірний)** | **Z (хаотичний)** |
| ----- | ------------------ | ---------------- | ----------------- |
| **A** | AX                 | AY               | AZ                |
| **B** | BX                 | BY               | BZ                |
| **C** | CX                 | CY               | CZ                |

---

## Інтерпретація комірок ABC+XYZ

### 1. **AX — стратегічні товари**

* Висока цінність і стабільний попит.
* Постійна наявність на складі.
* Пріоритетне управління.

---

### 2. **AY — важливі, але з коливаннями**

* Дає великий оборот, але попит нестабільний.
* Потрібен регулярний аналіз попиту та гнучкі закупівлі.

---

### 3. **AZ — важливі, але непередбачувані**

* Високий дохід, але попит хаотичний.
* Ризики переваги/дефіциту.
* Краще закуповувати по факту, за замовленнями.

---

### 4. **BX — середня цінність, стабільний попит**

* Добре прогнозуються.
* Можна закуповувати за планом.
* Рівень контролю трохи нижчий, ніж у AX.

---

### 5. **BY — помірні за значущістю і стабільністю**

* Стандартна стратегія управління запасами.
* Моніторинг, але не пріоритет.

---

### 6. **BZ — середня важливість, хаотичний попит**

* Бажано скоротити склад або перейти на замовлення під клієнта.

---

### 7. **CX — дешеві, але стабільні**

* Можна закуповувати великими партіями.
* Підходять для автоматичного поповнення.

---

### 8. **CY — дешеві, помірно нестабільні**

* Можливо мати запас, але обережно.
* Баланс між витратами та ризиками.

---

### 9. **CZ — дешеві і хаотичні**

* Кандидати на виведення з асортименту.
* Закупівля по факту або виключення.

---

## Як використовувати на практиці?

* **AX, BX** — автоматизація закупівель, регулярний перегляд.
* **AZ, BZ, CZ** — зменшення складських залишків, закупівля по замовленню.
* **CY, CZ** — кандидати на видалення/заміну.
* **ABC** — формує пріоритет за важливістю.
* **XYZ** — формує пріоритет за прогнозованістю.


# 🔷 Приклад

### ✅ A/X — **Золотий фонд**

* Високий внесок у виручку
* Стабільний попит
* **Що робити:**

  * Тримати максимальний пріоритет
  * Гарантувати постійну доступність
  * Ретельно відстежувати продажі
  * Не допускати out-of-stock за жодних умов
  * Застосовувати автоматичне поповнення

---

### ✅ A/Y — **Важливі, але коливаються**

* Висока цінність
* Середня стабільність
* **Що робити:**

  * Відстежувати тренди та сезонність
  * Використовувати просунутий прогноз попиту (наприклад, ковзні середні)
  * Тримати середній рівень запасів
  * Планувати промо, якщо спадання повторюється по часу

---

### ✅ A/Z — **Ризикові, але дорогі**

* Дуже важливі
* Дуже нестабільні
* **Що робити:**

  * Перевірити причини нестабільності: акції, збої постачання, сезонність
  * Застосувати підхід JIT (just in time)
  * Уникати переваги запасів
  * Узгодити зі стратегічним відділом: «тримати під замовлення» або «пульсова закупівля»

---

### 🟨 B/X — **Надійні робочі конячки**

* Середня цінність
* Надійний попит
* **Що робити:**

  * Підтримувати запаси стабільно
  * Можна автоматизувати закупівлі
  * Контролювати залишки, але без надмірностей

---

### 🟨 B/Y — **Менш стабільні, середня цінність**

* Складніше прогнозувати
* **Що робити:**

  * Використовувати помірне буферне зберігання
  * Відстежувати сплески
  * Застосовувати ручний контроль або ML-модель на основі коливань

---

### 🟨 B/Z — **Нестабільні та середньоцінні**

* Погано прогнозуються
* **Що робити:**

  * Мінімізувати запаси
  * Під замовлення, по заявці
  * Не тримати на складі «про запас»

---

### 🟥 C/X — **Багато, стабільно, але дешево**

* Мало приносять, але добре прогнозуються
* **Що робити:**

  * Використовувати автоматичне поповнення за залишковим принципом
  * Продавати через long-tail канали (маркетплейси, акції, розпродажі)
  * Можна формувати промо-набори

---

### 🟥 C/Y — **Багато, середньо передбачувано**

* Неефективні та нестабільні
* **Що робити:**

  * Провести аналіз — навіщо вони потрібні?
  * Скоротити номенклатуру
  * Передати на знижену ціну або розпродаж

---

### 🟥 C/Z — **Нижній баласт**

* Мінімальний внесок + хаотичний попит
* **Що робити:**

  * Оптимізувати: прибрати з зберігання
  * Списати, ліквідувати
  * Перевести на дозаказ за запитом клієнта
  * Не тримати на складі


In [19]:
df = pd.read_csv('https://raw.githubusercontent.com/cpython-projects/da_2603/refs/heads/main/abc_xyz_dataset.csv')
df.head()

,Item_ID,Item_Name,Category,Jan_Demand,Feb_Demand,Mar_Demand,Apr_Demand,May_Demand,Jun_Demand,Jul_Demand,Aug_Demand,Sep_Demand,Oct_Demand,Nov_Demand,Dec_Demand,Total_Annual_Units,Price_Per_Unit,Total_Sales_Value
0,ITM_001,Surface Near,Grocery,4516,4069,4664,4653,4508,4125,4669,4210,4824,4497,4259,4782,53776,10,537760
1,ITM_002,Central Him,Grocery,4792,4964,4628,4660,4897,5015,4805,4686,4896,4536,4520,5054,57453,100,5745300
2,ITM_003,Win Everyone,Apparel,61,175,38,43,15,161,224,41,387,340,70,21,1576,2,3152
3,ITM_004,Task Save,Apparel,1145,1113,717,832,783,954,1047,894,994,978,1136,712,11305,2,22610
4,ITM_005,Hotel Teacher,Grocery,1494,2051,1400,1918,1669,1733,1695,1560,1679,1381,1591,1662,19833,10,198330


In [20]:
df.columns.to_list()

['Item_ID',
 'Item_Name',
 'Category',
 'Jan_Demand',
 'Feb_Demand',
 'Mar_Demand',
 'Apr_Demand',
 'May_Demand',
 'Jun_Demand',
 'Jul_Demand',
 'Aug_Demand',
 'Sep_Demand',
 'Oct_Demand',
 'Nov_Demand',
 'Dec_Demand',
 'Total_Annual_Units',
 'Price_Per_Unit',
 'Total_Sales_Value']

In [23]:
df = df.sort_values(by='Total_Sales_Value', ascending=False).reset_index()

In [24]:
df

,index,Item_ID,Item_Name,Category,Jan_Demand,Feb_Demand,Mar_Demand,Apr_Demand,May_Demand,Jun_Demand,Jul_Demand,Aug_Demand,Sep_Demand,Oct_Demand,Nov_Demand,Dec_Demand,Total_Annual_Units,Price_Per_Unit,Total_Sales_Value
0,924,ITM_925,Ten And,Grocery,4991,4663,5068,4885,5127,4705,5130,5129,5127,4840,4774,5023,59462,1000,59462000
1,511,ITM_512,Radio Race,Grocery,4185,4087,3660,3774,4391,3906,3699,4155,3936,3646,3959,4208,47606,1000,47606000
2,521,ITM_522,Pattern Book,Grocery,3079,3185,3081,2794,3313,3329,3168,3241,3309,2997,2781,2859,37136,1000,37136000
3,394,ITM_395,Animal Key,Grocery,5180,4626,5010,4802,4856,5361,5504,4910,4631,4959,4867,4903,59609,500,29804500
4,168,ITM_169,Material Vote,Grocery,4871,4994,4740,4958,4428,4889,4754,4672,4477,4544,4553,4982,56862,500,28431000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,177,ITM_178,Investment Agreement,Electronics,63,63,59,72,55,65,60,59,59,61,75,65,756,2,1512
996,482,ITM_483,Along Have,Home & Kitchen,45,53,76,60,51,69,60,57,60,76,56,65,728,2,1456
997,408,ITM_409,Indicate Organization,Home & Kitchen,53,53,70,62,55,54,58,69,62,48,54,60,698,2,1396
998,568,ITM_569,Worry Because,Toys,50,65,61,60,51,55,48,51,54,55,50,45,645,2,1290


In [25]:
df['cum_sum'] = df.Total_Sales_Value.cumsum()
df

,index,Item_ID,Item_Name,Category,Jan_Demand,Feb_Demand,Mar_Demand,Apr_Demand,May_Demand,Jun_Demand,Jul_Demand,Aug_Demand,Sep_Demand,Oct_Demand,Nov_Demand,Dec_Demand,Total_Annual_Units,Price_Per_Unit,Total_Sales_Value,cum_sum
0,924,ITM_925,Ten And,Grocery,4991,4663,5068,4885,5127,4705,5130,5129,5127,4840,4774,5023,59462,1000,59462000,59462000
1,511,ITM_512,Radio Race,Grocery,4185,4087,3660,3774,4391,3906,3699,4155,3936,3646,3959,4208,47606,1000,47606000,107068000
2,521,ITM_522,Pattern Book,Grocery,3079,3185,3081,2794,3313,3329,3168,3241,3309,2997,2781,2859,37136,1000,37136000,144204000
3,394,ITM_395,Animal Key,Grocery,5180,4626,5010,4802,4856,5361,5504,4910,4631,4959,4867,4903,59609,500,29804500,174008500
4,168,ITM_169,Material Vote,Grocery,4871,4994,4740,4958,4428,4889,4754,4672,4477,4544,4553,4982,56862,500,28431000,202439500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,177,ITM_178,Investment Agreement,Electronics,63,63,59,72,55,65,60,59,59,61,75,65,756,2,1512,1072282530
996,482,ITM_483,Along Have,Home & Kitchen,45,53,76,60,51,69,60,57,60,76,56,65,728,2,1456,1072283986
997,408,ITM_409,Indicate Organization,Home & Kitchen,53,53,70,62,55,54,58,69,62,48,54,60,698,2,1396,1072285382
998,568,ITM_569,Worry Because,Toys,50,65,61,60,51,55,48,51,54,55,50,45,645,2,1290,1072286672


In [26]:
total = df.Total_Sales_Value.sum()

In [27]:
df['cum_sum_percents'] = df['cum_sum'] / total
df

,index,Item_ID,Item_Name,Category,Jan_Demand,Feb_Demand,Mar_Demand,Apr_Demand,May_Demand,Jun_Demand,...,Aug_Demand,Sep_Demand,Oct_Demand,Nov_Demand,Dec_Demand,Total_Annual_Units,Price_Per_Unit,Total_Sales_Value,cum_sum,cum_sum_percents
0,924,ITM_925,Ten And,Grocery,4991,4663,5068,4885,5127,4705,...,5129,5127,4840,4774,5023,59462,1000,59462000,59462000,0.055453
1,511,ITM_512,Radio Race,Grocery,4185,4087,3660,3774,4391,3906,...,4155,3936,3646,3959,4208,47606,1000,47606000,107068000,0.099850
2,521,ITM_522,Pattern Book,Grocery,3079,3185,3081,2794,3313,3329,...,3241,3309,2997,2781,2859,37136,1000,37136000,144204000,0.134483
3,394,ITM_395,Animal Key,Grocery,5180,4626,5010,4802,4856,5361,...,4910,4631,4959,4867,4903,59609,500,29804500,174008500,0.162278
4,168,ITM_169,Material Vote,Grocery,4871,4994,4740,4958,4428,4889,...,4672,4477,4544,4553,4982,56862,500,28431000,202439500,0.188792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,177,ITM_178,Investment Agreement,Electronics,63,63,59,72,55,65,...,59,59,61,75,65,756,2,1512,1072282530,0.999995
996,482,ITM_483,Along Have,Home & Kitchen,45,53,76,60,51,69,...,57,60,76,56,65,728,2,1456,1072283986,0.999996
997,408,ITM_409,Indicate Organization,Home & Kitchen,53,53,70,62,55,54,...,69,62,48,54,60,698,2,1396,1072285382,0.999998
998,568,ITM_569,Worry Because,Toys,50,65,61,60,51,55,...,51,54,55,50,45,645,2,1290,1072286672,0.999999


In [28]:
def abc_classify(item):
  if item <= 0.8:
    return 'A'
  if item <= 0.95:
    return 'B'
  return 'C'

df['ABC'] = df.cum_sum_percents.apply(abc_classify)
df

,index,Item_ID,Item_Name,Category,Jan_Demand,Feb_Demand,Mar_Demand,Apr_Demand,May_Demand,Jun_Demand,...,Sep_Demand,Oct_Demand,Nov_Demand,Dec_Demand,Total_Annual_Units,Price_Per_Unit,Total_Sales_Value,cum_sum,cum_sum_percents,ABC
0,924,ITM_925,Ten And,Grocery,4991,4663,5068,4885,5127,4705,...,5127,4840,4774,5023,59462,1000,59462000,59462000,0.055453,A
1,511,ITM_512,Radio Race,Grocery,4185,4087,3660,3774,4391,3906,...,3936,3646,3959,4208,47606,1000,47606000,107068000,0.099850,A
2,521,ITM_522,Pattern Book,Grocery,3079,3185,3081,2794,3313,3329,...,3309,2997,2781,2859,37136,1000,37136000,144204000,0.134483,A
3,394,ITM_395,Animal Key,Grocery,5180,4626,5010,4802,4856,5361,...,4631,4959,4867,4903,59609,500,29804500,174008500,0.162278,A
4,168,ITM_169,Material Vote,Grocery,4871,4994,4740,4958,4428,4889,...,4477,4544,4553,4982,56862,500,28431000,202439500,0.188792,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,177,ITM_178,Investment Agreement,Electronics,63,63,59,72,55,65,...,59,61,75,65,756,2,1512,1072282530,0.999995,C
996,482,ITM_483,Along Have,Home & Kitchen,45,53,76,60,51,69,...,60,76,56,65,728,2,1456,1072283986,0.999996,C
997,408,ITM_409,Indicate Organization,Home & Kitchen,53,53,70,62,55,54,...,62,48,54,60,698,2,1396,1072285382,0.999998,C
998,568,ITM_569,Worry Because,Toys,50,65,61,60,51,55,...,54,55,50,45,645,2,1290,1072286672,0.999999,C


In [32]:
df.ABC.value_counts()

,count
ABC,
C,743
B,170
A,87


In [34]:
month = [col for col in df.columns if '_Demand' in col]
month

['Jan_Demand',
 'Feb_Demand',
 'Mar_Demand',
 'Apr_Demand',
 'May_Demand',
 'Jun_Demand',
 'Jul_Demand',
 'Aug_Demand',
 'Sep_Demand',
 'Oct_Demand',
 'Nov_Demand',
 'Dec_Demand']

In [37]:
df['mean_sales'] = df[month].mean(axis=1)
df['std_sales'] = df[month].std(axis=1)

df['cv'] = df.std_sales / df.mean_sales
df

,index,Item_ID,Item_Name,Category,Jan_Demand,Feb_Demand,Mar_Demand,Apr_Demand,May_Demand,Jun_Demand,...,Price_Per_Unit,Total_Sales_Value,cum_sum,cum_sum_percents,ABC,mean,std,mean_sales,std_sales,cv
0,924,ITM_925,Ten And,Grocery,4991,4663,5068,4885,5127,4705,...,1000,59462000,59462000,0.055453,A,4955.166667,175.138300,4955.166667,175.138300,0.035345
1,511,ITM_512,Radio Race,Grocery,4185,4087,3660,3774,4391,3906,...,1000,47606000,107068000,0.099850,A,3967.166667,242.380930,3967.166667,242.380930,0.061097
2,521,ITM_522,Pattern Book,Grocery,3079,3185,3081,2794,3313,3329,...,1000,37136000,144204000,0.134483,A,3094.666667,199.492234,3094.666667,199.492234,0.064463
3,394,ITM_395,Animal Key,Grocery,5180,4626,5010,4802,4856,5361,...,500,29804500,174008500,0.162278,A,4967.416667,265.766972,4967.416667,265.766972,0.053502
4,168,ITM_169,Material Vote,Grocery,4871,4994,4740,4958,4428,4889,...,500,28431000,202439500,0.188792,A,4738.500000,203.303848,4738.500000,203.303848,0.042905
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,177,ITM_178,Investment Agreement,Electronics,63,63,59,72,55,65,...,2,1512,1072282530,0.999995,C,63.000000,5.704862,63.000000,5.704862,0.090553
996,482,ITM_483,Along Have,Home & Kitchen,45,53,76,60,51,69,...,2,1456,1072283986,0.999996,C,60.666667,9.499601,60.666667,9.499601,0.156587
997,408,ITM_409,Indicate Organization,Home & Kitchen,53,53,70,62,55,54,...,2,1396,1072285382,0.999998,C,58.166667,6.685579,58.166667,6.685579,0.114938
998,568,ITM_569,Worry Because,Toys,50,65,61,60,51,55,...,2,1290,1072286672,0.999999,C,53.750000,5.832900,53.750000,5.832900,0.108519


In [40]:
def xyz_classify(item):
  if item <= 0.25:
    return 'X'
  if item <= 0.50:
    return 'Y'
  return 'Z'


df['XYZ'] = df.cv.apply(xyz_classify)
df

,index,Item_ID,Item_Name,Category,Jan_Demand,Feb_Demand,Mar_Demand,Apr_Demand,May_Demand,Jun_Demand,...,cum_sum,cum_sum_percents,ABC,mean,std,mean_sales,std_sales,cv,xyz,XYZ
0,924,ITM_925,Ten And,Grocery,4991,4663,5068,4885,5127,4705,...,59462000,0.055453,A,4955.166667,175.138300,4955.166667,175.138300,0.035345,X,X
1,511,ITM_512,Radio Race,Grocery,4185,4087,3660,3774,4391,3906,...,107068000,0.099850,A,3967.166667,242.380930,3967.166667,242.380930,0.061097,X,X
2,521,ITM_522,Pattern Book,Grocery,3079,3185,3081,2794,3313,3329,...,144204000,0.134483,A,3094.666667,199.492234,3094.666667,199.492234,0.064463,X,X
3,394,ITM_395,Animal Key,Grocery,5180,4626,5010,4802,4856,5361,...,174008500,0.162278,A,4967.416667,265.766972,4967.416667,265.766972,0.053502,X,X
4,168,ITM_169,Material Vote,Grocery,4871,4994,4740,4958,4428,4889,...,202439500,0.188792,A,4738.500000,203.303848,4738.500000,203.303848,0.042905,X,X
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,177,ITM_178,Investment Agreement,Electronics,63,63,59,72,55,65,...,1072282530,0.999995,C,63.000000,5.704862,63.000000,5.704862,0.090553,X,X
996,482,ITM_483,Along Have,Home & Kitchen,45,53,76,60,51,69,...,1072283986,0.999996,C,60.666667,9.499601,60.666667,9.499601,0.156587,X,X
997,408,ITM_409,Indicate Organization,Home & Kitchen,53,53,70,62,55,54,...,1072285382,0.999998,C,58.166667,6.685579,58.166667,6.685579,0.114938,X,X
998,568,ITM_569,Worry Because,Toys,50,65,61,60,51,55,...,1072286672,0.999999,C,53.750000,5.832900,53.750000,5.832900,0.108519,X,X


In [41]:
df.XYZ.value_counts()

,count
XYZ,
X,853
Y,86
Z,61


In [42]:
df['ABC_XYZ'] = df.ABC + df.XYZ
df

,index,Item_ID,Item_Name,Category,Jan_Demand,Feb_Demand,Mar_Demand,Apr_Demand,May_Demand,Jun_Demand,...,cum_sum_percents,ABC,mean,std,mean_sales,std_sales,cv,xyz,XYZ,ABC_XYZ
0,924,ITM_925,Ten And,Grocery,4991,4663,5068,4885,5127,4705,...,0.055453,A,4955.166667,175.138300,4955.166667,175.138300,0.035345,X,X,AX
1,511,ITM_512,Radio Race,Grocery,4185,4087,3660,3774,4391,3906,...,0.099850,A,3967.166667,242.380930,3967.166667,242.380930,0.061097,X,X,AX
2,521,ITM_522,Pattern Book,Grocery,3079,3185,3081,2794,3313,3329,...,0.134483,A,3094.666667,199.492234,3094.666667,199.492234,0.064463,X,X,AX
3,394,ITM_395,Animal Key,Grocery,5180,4626,5010,4802,4856,5361,...,0.162278,A,4967.416667,265.766972,4967.416667,265.766972,0.053502,X,X,AX
4,168,ITM_169,Material Vote,Grocery,4871,4994,4740,4958,4428,4889,...,0.188792,A,4738.500000,203.303848,4738.500000,203.303848,0.042905,X,X,AX
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,177,ITM_178,Investment Agreement,Electronics,63,63,59,72,55,65,...,0.999995,C,63.000000,5.704862,63.000000,5.704862,0.090553,X,X,CX
996,482,ITM_483,Along Have,Home & Kitchen,45,53,76,60,51,69,...,0.999996,C,60.666667,9.499601,60.666667,9.499601,0.156587,X,X,CX
997,408,ITM_409,Indicate Organization,Home & Kitchen,53,53,70,62,55,54,...,0.999998,C,58.166667,6.685579,58.166667,6.685579,0.114938,X,X,CX
998,568,ITM_569,Worry Because,Toys,50,65,61,60,51,55,...,0.999999,C,53.750000,5.832900,53.750000,5.832900,0.108519,X,X,CX


In [44]:
df.ABC_XYZ.value_counts()

,count
ABC_XYZ,
CX,615
BX,154
AX,84
CY,72
CZ,56
BY,11
BZ,5
AY,3


In [45]:
df.shape

(1000, 30)